In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
MODEL_DATA_DIR = PROJECT_ROOT / "data" / "model_ready"
EVENT_CLEAN_DIR = PROJECT_ROOT / "data" / "event_data" / "clean"

class WinProbabilityLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                             num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits.squeeze(-1)

checkpoint = torch.load(MODEL_DIR / "win_probability_lstm_v1.pt")
model = WinProbabilityLSTM(input_size=checkpoint['input_size'], hidden_size=checkpoint['hidden_size'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

feature_cols = checkpoint['feature_cols']

data = torch.load(MODEL_DATA_DIR / "sequences.pt")
X_test, len_test, y_test, keys_test = data['X_test'], data['len_test'], data['y_test'], data['keys_test']

deaths = pd.read_parquet(EVENT_CLEAN_DIR / "deaths_clean.parquet")
objectives = pd.read_parquet(EVENT_CLEAN_DIR / "objectives_clean.parquet")
mid_boss = pd.read_parquet(EVENT_CLEAN_DIR / "mid_boss_clean.parquet")

print("Reloaded model and data successfully")

Reloaded model and data successfully


In [5]:
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
full_features = pd.read_parquet(FEATURE_DIR / "full_features_mirrored.parquet")

match_ids = full_features['match_id'].unique()
rng = np.random.default_rng(seed=42)
rng.shuffle(match_ids)

n = len(match_ids)
train_ids = match_ids[:int(n * 0.7)]
val_ids = match_ids[int(n * 0.7):int(n * 0.85)]
test_ids = match_ids[int(n * 0.85):]

test_df = full_features[full_features['match_id'].isin(test_ids)].copy()
print(f"test_df shape: {test_df.shape}")

test_df shape: (26900, 27)


In [12]:
def find_biggest_swing(X, lengths, keys, model):
    model.eval()
    with torch.no_grad():
        logits = model(X)
        probs = torch.sigmoid(logits)

    swings = []
    for i in range(X.shape[0]):
        L = lengths[i].item()
        if L < 3:
            continue
        seq_probs = probs[i, :L]
        max_drop = 0.0
        max_drop_t = None
        for t in range(1, L):
            drop = seq_probs[t-1].item() - seq_probs[t].item()
            if drop > max_drop:
                max_drop = drop
                max_drop_t = t
        swings.append({
            'idx': i,
            'match_id': keys[i][0],
            'perspective': keys[i][1],
            'max_drop': max_drop,
            'drop_at_timestep': max_drop_t,
            'length': L,
        })

    return pd.DataFrame(swings).sort_values('max_drop', ascending=False)

swings_df = find_biggest_swing(X_test, len_test, keys_test, model)
swings_df.head(10)

,idx,match_id,perspective,max_drop,drop_at_timestep,length
428,428,22637391,Team0,0.690509,6.0,11
2629,2629,26601874,Team1,0.669625,8.0,10
2521,2521,26462291,Team1,0.669536,6.0,9
19,19,16521725,Team1,0.666233,5.0,6
1013,1013,23592863,Team1,0.656212,10.0,11
1660,1660,25526759,Team0,0.655648,5.0,8
982,982,23588753,Team0,0.654735,3.0,11
2649,2649,26602656,Team1,0.654579,8.0,10
708,708,22878931,Team0,0.649923,7.0,9
1592,1592,25418935,Team0,0.645092,8.0,12


In [14]:
# one match to prototype with - picked one where probabilities fluctuate

sample_match_id, sample_perspective = 22637391, 'Team0'
sample_idx = swings_df.iloc[0]['idx']

sample_X = X_test[sample_idx:sample_idx+1]
sample_len = len_test[sample_idx].item()
sample_y = y_test[sample_idx].item()

print(f"Match {sample_match_id} | Perspective {sample_perspective}")
print(f"Real length: {sample_len} timesteps")
print(f"Actual outcome (won): {sample_y}")

model.eval()
with torch.no_grad():
    logits = model(sample_X)
    probs = torch.sigmoid(logits).squeeze(0)

real_probs = probs[:sample_len]

match_rows = test_df[
    (test_df['match_id'] == sample_match_id) & (test_df['perspective'] == sample_perspective)
].sort_values('game_time_s')
game_times = match_rows['game_time_s'].values

print("\nTimestep -> game_time_s -> win_prob:")
for t, (gt, p) in enumerate(zip(game_times, real_probs)):
    print(f"  timestep {t}: game_time_s={gt}, win_prob={p.item():.4f}")

Match 22637391 | Perspective Team0
Real length: 11 timesteps
Actual outcome (won): 0.0

Timestep -> game_time_s -> win_prob:
  timestep 0: game_time_s=180, win_prob=0.2129
  timestep 1: game_time_s=360, win_prob=0.5025
  timestep 2: game_time_s=540, win_prob=0.8261
  timestep 3: game_time_s=720, win_prob=0.7043
  timestep 4: game_time_s=900, win_prob=0.8881
  timestep 5: game_time_s=1200, win_prob=0.8892
  timestep 6: game_time_s=1500, win_prob=0.1987
  timestep 7: game_time_s=1800, win_prob=0.2021
  timestep 8: game_time_s=2100, win_prob=0.7447
  timestep 9: game_time_s=2400, win_prob=0.4763
  timestep 10: game_time_s=2823, win_prob=0.1553


In [16]:
window_start = 1200
window_end = 1500

own_team = sample_perspective
enemy_team = 'Team1' if own_team == 'Team0' else 'Team0'

print(f"Match {sample_match_id} | Own team: {own_team} | Enemy team: {enemy_team}")
print(f"Window: {window_start}s - {window_end}s\n")

match_deaths = deaths[
    (deaths['match_id'] == sample_match_id) &
    (deaths['game_time_s'] >= window_start) &
    (deaths['game_time_s'] <= window_end)
].sort_values('game_time_s')
print("Deaths in window:")
print(match_deaths[['team', 'game_time_s', 'hero_id', 'killer_player_slot']])

match_objectives = objectives[
    (objectives['match_id'] == sample_match_id) &
    (objectives['destroyed_time_s'] >= window_start) &
    (objectives['destroyed_time_s'] <= window_end)
].sort_values('destroyed_time_s')
print("\nObjectives destroyed in window:")
print(match_objectives[['team', 'team_objective', 'destroyed_time_s']])

match_mid_boss = mid_boss[
    (mid_boss['match_id'] == sample_match_id) &
    (mid_boss['destroyed_time_s'] >= window_start) &
    (mid_boss['destroyed_time_s'] <= window_end)
].sort_values('destroyed_time_s')
print("\nMid boss claimed in window:")
print(match_mid_boss[['team_claimed', 'destroyed_time_s']])

Match 22637391 | Own team: Team0 | Enemy team: Team1
Window: 1200s - 1500s

Deaths in window:
         team  game_time_s  hero_id  killer_player_slot
474176  Team0         1206       17                  10
474201  Team0         1234       13                   7
474166  Team1         1353        1                   3
474151  Team0         1391        8                  10
474177  Team0         1395       17                  10
474141  Team0         1473        7                  12
474184  Team0         1476        3                  11

Objectives destroyed in window:
         team team_objective  destroyed_time_s
149497  Team1     Tier2Lane1              1300
149493  Team0     Tier2Lane1              1382

Mid boss claimed in window:
Empty DataFrame
Columns: [team_claimed, destroyed_time_s]
Index: []


In [25]:
def format_net_worth(nw_diff, own_total, enemy_total):
    total_economy = own_total + enemy_total
    pct = (abs(nw_diff) / total_economy * 100) if total_economy > 0 else 0

    if nw_diff > 0:
        return f"You lead by {abs(nw_diff):,.0f} souls ({pct:.1f}% of total match economy)"
    elif nw_diff < 0:
        return f"Enemy leads by {abs(nw_diff):,.0f} souls ({pct:.1f}% of total match economy)"
    else:
        return "Souls are even"

In [28]:
def explain_match_v3(match_id, perspective, X, lengths, keys, model, test_df, deaths_df, objectives_df, mid_boss_df, feature_cols, top_n=3):
    idx = keys.index((match_id, perspective))
    seq_len = lengths[idx].item()

    model.eval()
    with torch.no_grad():
        logits = model(X[idx:idx+1])
        probs = torch.sigmoid(logits).squeeze(0)[:seq_len]

    match_rows = test_df[
        (test_df['match_id'] == match_id) & (test_df['perspective'] == perspective)
    ].sort_values('game_time_s')
    game_times = match_rows['game_time_s'].values
    net_worth_diffs = match_rows['net_worth_diff'].values

    own_team = perspective
    enemy_team = 'Team1' if own_team == 'Team0' else 'Team0'

    # pick the correct raw net worth columns depending on which literal team is "own"
    if own_team == 'Team0':
        own_nw = match_rows['team_net_worth_team0'].values
        enemy_nw = match_rows['team_net_worth_team1'].values
    else:
        own_nw = match_rows['team_net_worth_team1'].values
        enemy_nw = match_rows['team_net_worth_team0'].values

    drops = []
    for t in range(1, seq_len):
        delta = probs[t].item() - probs[t-1].item()
        drops.append({
            'from_t': t - 1, 'to_t': t,
            'from_time': int(game_times[t-1]), 'to_time': int(game_times[t]),
            'from_prob': probs[t-1].item(), 'to_prob': probs[t].item(),
            'delta': delta,
            'from_nw_diff': net_worth_diffs[t-1], 'to_nw_diff': net_worth_diffs[t],
            'from_own_nw': own_nw[t-1], 'from_enemy_nw': enemy_nw[t-1],
            'to_own_nw': own_nw[t], 'to_enemy_nw': enemy_nw[t],
        })
    drops_df = pd.DataFrame(drops)
    biggest_drops = drops_df.sort_values('delta').head(top_n)

    print(f"=== Match {match_id} | Perspective: {perspective} ===")
    print(f"Final outcome: {'WON' if match_rows['won'].iloc[0] else 'LOST'}\n")

    for _, row in biggest_drops.iterrows():
        window_start, window_end = int(row['from_time']), int(row['to_time'])

        d = deaths_df[
            (deaths_df['match_id'] == match_id) &
            (deaths_df['game_time_s'] >= window_start) &
            (deaths_df['game_time_s'] <= window_end)
        ]
        own_deaths = (d['team'] == own_team).sum()
        enemy_deaths = (d['team'] == enemy_team).sum()

        o = objectives_df[
            (objectives_df['match_id'] == match_id) &
            (objectives_df['destroyed_time_s'] >= window_start) &
            (objectives_df['destroyed_time_s'] <= window_end)
        ]
        own_obj = (o['team'] == own_team).sum()
        enemy_obj = (o['team'] == enemy_team).sum()

        mb = mid_boss_df[
            (mid_boss_df['match_id'] == match_id) &
            (mid_boss_df['destroyed_time_s'] >= window_start) &
            (mid_boss_df['destroyed_time_s'] <= window_end)
        ]
        enemy_mb = (mb['team_claimed'] == enemy_team).sum()
        own_mb = (mb['team_claimed'] == own_team).sum()

        mins_start, secs_start = divmod(window_start, 60)
        mins_end, secs_end = divmod(window_end, 60)

        print(f"--- Turning point: {mins_start}:{secs_start:02d} → {mins_end}:{secs_end:02d} ---")
        print(f"Win probability: {row['from_prob']*100:.1f}% → {row['to_prob']*100:.1f}% ({row['delta']*100:+.1f} pts)")
        print(f"Soul lead: {format_net_worth(row['from_nw_diff'], row['from_own_nw'], row['from_enemy_nw'])}")
        print(f"       →  {format_net_worth(row['to_nw_diff'], row['to_own_nw'], row['to_enemy_nw'])}")

        narrative_parts = []
        if own_deaths > 0 or enemy_deaths > 0:
            narrative_parts.append(f"Deaths: you {own_deaths}, enemy {enemy_deaths}.")
        if own_obj > 0 or enemy_obj > 0:
            narrative_parts.append(f"Objectives: you {own_obj}, enemy {enemy_obj}.")
        if own_mb > 0 or enemy_mb > 0:
            narrative_parts.append(f"Mid boss claimed by: {'you' if own_mb else 'enemy'}.")

        if narrative_parts:
            print(" ".join(narrative_parts))
        print()

explain_match_v3(22637391, 'Team0', X_test, len_test, keys_test, model, test_df, deaths, objectives, mid_boss, feature_cols)

=== Match 22637391 | Perspective: Team0 ===
Final outcome: LOST

--- Turning point: 20:00 → 25:00 ---
Win probability: 88.9% → 19.9% (-69.1 pts)
Soul lead: You lead by 5,788 souls (3.4% of total match economy)
       →  Enemy leads by 9,782 souls (4.1% of total match economy)
Deaths: you 6, enemy 1. Objectives: you 1, enemy 1.

--- Turning point: 40:00 → 47:03 ---
Win probability: 47.6% → 15.5% (-32.1 pts)
Soul lead: Enemy leads by 3,659 souls (0.7% of total match economy)
       →  Enemy leads by 11,514 souls (1.8% of total match economy)
Deaths: you 12, enemy 10. Objectives: you 3, enemy 3.

--- Turning point: 35:00 → 40:00 ---
Win probability: 74.5% → 47.6% (-26.8 pts)
Soul lead: You lead by 2,785 souls (0.7% of total match economy)
       →  Enemy leads by 3,659 souls (0.7% of total match economy)
Deaths: you 9, enemy 10. Objectives: you 1, enemy 1. Mid boss claimed by: you.



In [34]:
def find_event_chain_v3(match_id, own_team, window_start, window_end, deaths_df, objectives_df, mid_boss_df, lookback_s=90):
    enemy_team = 'Team1' if own_team == 'Team0' else 'Team0'

    d = deaths_df[
        (deaths_df['match_id'] == match_id) &
        (deaths_df['game_time_s'] >= window_start) &
        (deaths_df['game_time_s'] <= window_end)
    ].sort_values('game_time_s')
    own_deaths = d[d['team'] == own_team]

    o = objectives_df[
        (objectives_df['match_id'] == match_id) &
        (objectives_df['team'] == enemy_team) &
        (objectives_df['destroyed_time_s'] >= window_start) &
        (objectives_df['destroyed_time_s'] <= window_end + lookback_s)
    ].sort_values('destroyed_time_s')

    mb = mid_boss_df[
        (mid_boss_df['match_id'] == match_id) &
        (mid_boss_df['team_claimed'] == enemy_team) &
        (mid_boss_df['destroyed_time_s'] >= window_start) &
        (mid_boss_df['destroyed_time_s'] <= window_end + lookback_s)
    ]

    events = []
    for _, obj in o.iterrows():
        events.append({'time': obj['destroyed_time_s'], 'type': 'objective', 'name': obj['team_objective']})
    for _, m in mb.iterrows():
        events.append({'time': m['destroyed_time_s'], 'type': 'mid_boss', 'name': 'Mid Boss'})

    chains = []
    for event in events:
        preceding = own_deaths[
            (own_deaths['game_time_s'] <= event['time']) &
            (own_deaths['game_time_s'] >= event['time'] - lookback_s)
        ]
        if len(preceding) == 0:
            continue

        closest_death = preceding.iloc[-1]
        death_time = closest_death['game_time_s']
        death_duration = closest_death['death_duration_s']
        respawn_time = death_time + death_duration
        gap_s = event['time'] - death_time

        was_dead_during_event = event['time'] <= respawn_time
        grace_period_s = 20
        plausibly_absent = event['time'] <= (respawn_time + grace_period_s)

        chains.append({
            'death_time': int(death_time),
            'hero_id': int(closest_death['hero_id']),
            'death_duration_s': int(death_duration),
            'respawn_time': int(respawn_time),
            'event_type': event['type'],
            'event_name': event['name'],
            'event_time': int(event['time']),
            'gap_s': int(gap_s),
            'was_dead_during_event': bool(was_dead_during_event),
            'plausibly_absent': bool(plausibly_absent),
        })

    return {
        'own_death_count': len(own_deaths),
        'enemy_death_count': int((d['team'] == enemy_team).sum()),
        'chains': chains,
    }

result = find_event_chain_v3(22637391, 'Team0', 1200, 1500, deaths, objectives, mid_boss)
result

{'own_death_count': 6,
 'enemy_death_count': 1,
 'chains': [{'death_time': 1234,
   'hero_id': 13,
   'death_duration_s': 46,
   'respawn_time': 1280,
   'event_type': 'objective',
   'event_name': 'Tier2Lane1',
   'event_time': 1300,
   'gap_s': 66,
   'was_dead_during_event': False,
   'plausibly_absent': True},
  {'death_time': 1476,
   'hero_id': 3,
   'death_duration_s': 53,
   'respawn_time': 1529,
   'event_type': 'objective',
   'event_name': 'Tier2Lane3',
   'event_time': 1511,
   'gap_s': 35,
   'was_dead_during_event': True,
   'plausibly_absent': True}]}

In [40]:
def format_objective_name(raw_name):
    if raw_name.startswith('Tier1Lane'):
        lane = raw_name[-1]
        return f"Guardian (Lane {lane})"
    elif raw_name.startswith('Tier2Lane'):
        lane = raw_name[-1]
        return f"Walker (Lane {lane})"
    elif raw_name.startswith('BarrackBossLane'):
        lane = raw_name[-1]
        return f"Base Guardian (Lane {lane})"
    elif raw_name.startswith('TitanShieldGenerator'):
        return "Shrine"
    elif raw_name == 'Titan':
        return "Patron phase1"
    elif raw_name == 'Core':
        return "Patron phase2"
    else:
        return raw_name

for name in objectives['team_objective'].unique():
    print(f"{name} -> {format_objective_name(name)}")

Tier1Lane3 -> Guardian (Lane 3)
Tier1Lane4 -> Guardian (Lane 4)
Tier1Lane2 -> Guardian (Lane 2)
Tier2Lane4 -> Walker (Lane 4)
Tier1Lane1 -> Guardian (Lane 1)
Tier2Lane1 -> Walker (Lane 1)
Tier2Lane2 -> Walker (Lane 2)
Tier2Lane3 -> Walker (Lane 3)
BarrackBossLane3 -> Base Guardian (Lane 3)
TitanShieldGenerator1 -> Shrine
BarrackBossLane4 -> Base Guardian (Lane 4)
BarrackBossLane2 -> Base Guardian (Lane 2)
TitanShieldGenerator2 -> Shrine
Titan -> Patron phase1
BarrackBossLane1 -> Base Guardian (Lane 1)
Core -> Patron phase2


In [44]:
def narrate_chain(chain):
    mins, secs = divmod(chain['death_time'], 60)
    event_mins, event_secs = divmod(chain['event_time'], 60)
    event_label = "Mid Boss" if chain['event_type'] == 'mid_boss' else format_objective_name(chain['event_name'])

    if chain['was_dead_during_event']:
        return (f"Your teammate died at {mins}:{secs:02d}. While they were still respawning, "
                f"the enemy claimed {event_label} at {event_mins}:{event_secs:02d} "
                f"({chain['gap_s']}s later).")
    elif chain['plausibly_absent']:
        return (f"Your teammate died at {mins}:{secs:02d}. Shortly after respawning, "
                f"the enemy claimed {event_label} at {event_mins}:{event_secs:02d} "
                f"({chain['gap_s']}s after the death) — the death may have left your team short-handed.")
    else:
        return None

for chain in result['chains']:
    sentence = narrate_chain(chain)
    if sentence:
        print(sentence)
        print()

Your teammate died at 20:34. Shortly after respawning, the enemy claimed Walker (Lane 1) at 21:40 (66s after the death) — the death may have left your team short-handed.

Your teammate died at 24:36. While they were still respawning, the enemy claimed Walker (Lane 3) at 25:11 (35s later).



In [45]:
def explain_match_final(match_id, perspective, X, lengths, keys, model, test_df,
                          deaths_df, objectives_df, mid_boss_df, feature_cols,
                          top_n=3, lookback_s=90):
    idx = keys.index((match_id, perspective))
    seq_len = lengths[idx].item()

    model.eval()
    with torch.no_grad():
        logits = model(X[idx:idx+1])
        probs = torch.sigmoid(logits).squeeze(0)[:seq_len]

    match_rows = test_df[
        (test_df['match_id'] == match_id) & (test_df['perspective'] == perspective)
    ].sort_values('game_time_s')
    game_times = match_rows['game_time_s'].values
    net_worth_diffs = match_rows['net_worth_diff'].values

    own_team = perspective
    enemy_team = 'Team1' if own_team == 'Team0' else 'Team0'

    if own_team == 'Team0':
        own_nw = match_rows['team_net_worth_team0'].values
        enemy_nw = match_rows['team_net_worth_team1'].values
    else:
        own_nw = match_rows['team_net_worth_team1'].values
        enemy_nw = match_rows['team_net_worth_team0'].values

    # find biggest probability drops
    drops = []
    for t in range(1, seq_len):
        delta = probs[t].item() - probs[t-1].item()
        drops.append({
            'from_time': int(game_times[t-1]), 'to_time': int(game_times[t]),
            'from_prob': probs[t-1].item(), 'to_prob': probs[t].item(),
            'delta': delta,
            'from_nw_diff': net_worth_diffs[t-1], 'to_nw_diff': net_worth_diffs[t],
            'from_own_nw': own_nw[t-1], 'from_enemy_nw': enemy_nw[t-1],
            'to_own_nw': own_nw[t], 'to_enemy_nw': enemy_nw[t],
        })
    drops_df = pd.DataFrame(drops)
    biggest_drops = drops_df.sort_values('delta').head(top_n)

    outcome = 'WON' if match_rows['won'].iloc[0] else 'LOST'
    print("=" * 60)
    print(f"MATCH REPORT: {match_id} | Your perspective: {perspective}")
    print(f"Final outcome: {outcome}")
    print("=" * 60)

    for i, (_, row) in enumerate(biggest_drops.iterrows(), 1):
        window_start, window_end = int(row['from_time']), int(row['to_time'])

        d = deaths_df[
            (deaths_df['match_id'] == match_id) &
            (deaths_df['game_time_s'] >= window_start) &
            (deaths_df['game_time_s'] <= window_end)
        ]
        own_deaths_count = (d['team'] == own_team).sum()
        enemy_deaths_count = (d['team'] == enemy_team).sum()

        o = objectives_df[
            (objectives_df['match_id'] == match_id) &
            (objectives_df['destroyed_time_s'] >= window_start) &
            (objectives_df['destroyed_time_s'] <= window_end)
        ]
        own_obj_count = (o['team'] == own_team).sum()
        enemy_obj_count = (o['team'] == enemy_team).sum()

        mb = mid_boss_df[
            (mid_boss_df['match_id'] == match_id) &
            (mid_boss_df['destroyed_time_s'] >= window_start) &
            (mid_boss_df['destroyed_time_s'] <= window_end)
        ]
        enemy_mb_count = (mb['team_claimed'] == enemy_team).sum()
        own_mb_count = (mb['team_claimed'] == own_team).sum()

        mins_start, secs_start = divmod(window_start, 60)
        mins_end, secs_end = divmod(window_end, 60)

        print(f"\n--- Turning Point #{i}: {mins_start}:{secs_start:02d} → {mins_end}:{secs_end:02d} ---")
        print(f"Win probability: {row['from_prob']*100:.1f}% → {row['to_prob']*100:.1f}% ({row['delta']*100:+.1f} pts)")
        print(f"Soul lead: {format_net_worth(row['from_nw_diff'], row['from_own_nw'], row['from_enemy_nw'])}")
        print(f"       →  {format_net_worth(row['to_nw_diff'], row['to_own_nw'], row['to_enemy_nw'])}")

        summary_parts = []
        if own_deaths_count > 0 or enemy_deaths_count > 0:
            summary_parts.append(f"Deaths: you {own_deaths_count}, enemy {enemy_deaths_count}.")
        if own_obj_count > 0 or enemy_obj_count > 0:
            summary_parts.append(f"Objectives: you {own_obj_count}, enemy {enemy_obj_count}.")
        if own_mb_count > 0 or enemy_mb_count > 0:
            summary_parts.append(f"Mid boss claimed by: {'you' if own_mb_count else 'enemy'}.")
        if summary_parts:
            print(" ".join(summary_parts))

        # event chains within this window
        chain_result = find_event_chain_v3(match_id, own_team, window_start, window_end,
                                             deaths_df, objectives_df, mid_boss_df, lookback_s)
        narrated_any = False
        for chain in chain_result['chains']:
            sentence = narrate_chain(chain)
            if sentence:
                print(f"  → {sentence}")
                narrated_any = True

        if not narrated_any and not summary_parts:
            print("No major recorded events in this window — likely a gradual economy shift.")

    print("\n" + "=" * 60)


explain_match_final(22637391, 'Team0', X_test, len_test, keys_test, model, test_df,
                     deaths, objectives, mid_boss, feature_cols)

MATCH REPORT: 22637391 | Your perspective: Team0
Final outcome: LOST

--- Turning Point #1: 20:00 → 25:00 ---
Win probability: 88.9% → 19.9% (-69.1 pts)
Soul lead: You lead by 5,788 souls (3.4% of total match economy)
       →  Enemy leads by 9,782 souls (4.1% of total match economy)
Deaths: you 6, enemy 1. Objectives: you 1, enemy 1.
  → Your teammate died at 20:34. Shortly after respawning, the enemy claimed Walker (Lane 1) at 21:40 (66s after the death) — the death may have left your team short-handed.
  → Your teammate died at 24:36. While they were still respawning, the enemy claimed Walker (Lane 3) at 25:11 (35s later).

--- Turning Point #2: 40:00 → 47:03 ---
Win probability: 47.6% → 15.5% (-32.1 pts)
Soul lead: Enemy leads by 3,659 souls (0.7% of total match economy)
       →  Enemy leads by 11,514 souls (1.8% of total match economy)
Deaths: you 12, enemy 10. Objectives: you 3, enemy 3.
  → Your teammate died at 41:29. While they were still respawning, the enemy claimed Base Gu